In [ ]:

# =========================
# Supervised Fine-tune on small labeled target subset
# Steps:
#   Load the trained PlantVillage baseline checkpoint
#   Fine-tune ONLY on PlantDoc/train_labeled
#   Evaluate on PlantDoc/test
# =========================

import os
import time
import json
import random
import shutil
from typing import Dict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# =========================
# Paths and config
# =========================

# Drive paths
DRIVE_ROOT = "/content/drive/MyDrive/dataset"
BASELINE_CKPT = "/content/drive/MyDrive/plant_disease_project/runs_supervised_baseline/best.pt"

# Only these two PlantDoc folders are needed in this notebook
PD_LABELED_DIR = f"{DRIVE_ROOT}/PlantDoc/train_labeled"
PD_TEST_DIR = f"{DRIVE_ROOT}/PlantDoc/test"

OUT_DIR = "/content/drive/MyDrive/plant_disease_project/runs_finetune_supervised_cleaned"
os.makedirs(OUT_DIR, exist_ok=True)

# Fine-tune config
MODEL_NAME = "resnet18"          # must match baseline model
EPOCHS_FINETUNE = 8
LR_FINETUNE = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 64
NUM_WORKERS = 2
USE_AMP = True
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[Info] device =", device)


[Info] device = cuda


In [6]:

# =========================
# Sanity checks + remove empty folders from train_labeled
# =========================
if not os.path.isfile(BASELINE_CKPT):
    raise RuntimeError(f"[ERROR] Baseline checkpoint not found: {BASELINE_CKPT}")

if not os.path.isdir(PD_LABELED_DIR):
    raise RuntimeError(f"[ERROR] train_labeled folder not found: {PD_LABELED_DIR}")

if not os.path.isdir(PD_TEST_DIR):
    raise RuntimeError(f"[ERROR] test folder not found: {PD_TEST_DIR}")

empty_dirs = []
for c in os.listdir(PD_LABELED_DIR):
    p = os.path.join(PD_LABELED_DIR, c)
    if os.path.isdir(p) and len(os.listdir(p)) == 0:
        empty_dirs.append(p)

for p in empty_dirs:
    print("[Removing empty folder]", p)
    shutil.rmtree(p)

print(f"[Info] Removed {len(empty_dirs)} empty folders from train_labeled")


[Info] Removed 0 empty folders from train_labeled


In [7]:

# =========================
# Load baseline checkpoint
# =========================
ckpt = torch.load(BASELINE_CKPT, map_location="cpu")
print("[Info] checkpoint keys:", ckpt.keys())

if "classes" not in ckpt:
    raise RuntimeError("Baseline checkpoint does not contain 'classes'. Please save classes in baseline best.pt.")

source_classes = ckpt["classes"]
num_classes = len(source_classes)
source_class_to_idx = {name: i for i, name in enumerate(source_classes)}

print("[Info] num_classes =", num_classes)
print("[Info] source classes =", source_classes)


[Info] checkpoint keys: dict_keys(['model_state_dict', 'epoch', 'best_acc', 'classes', 'model_name'])
[Info] num_classes = 15
[Info] source classes = ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [8]:

# =========================
# Transforms
# =========================
normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    normalize,
])

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])


In [9]:

# =========================
# PlantVillage -> PlantDoc class mapping
# Update names here ONLY if your folder names differ
# =========================
PV2PD = {
    "Pepper__bell___Bacterial_spot": "Bell_pepper leaf spot",
    "Pepper__bell___healthy": "Bell_pepper leaf",

    "Potato___Early_blight": "Potato leaf early blight",
    "Potato___Late_blight": "Potato leaf late blight",
    "Potato___healthy": None,

    "Tomato_Bacterial_spot": "Tomato leaf bacterial spot",
    "Tomato_Early_blight": "Tomato Early blight leaf",
    "Tomato_Late_blight": "Tomato leaf late blight",
    "Tomato_Leaf_Mold": "Tomato mold leaf",
    "Tomato_Septoria_leaf_spot": "Tomato Septoria leaf spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Tomato two spotted spider mites leaf",
    "Tomato__Target_Spot": None,
    "Tomato__Tomato_YellowLeaf__Curl_Virus": "Tomato leaf yellow virus",
    "Tomato__Tomato_mosaic_virus": "Tomato leaf mosaic virus",
    "Tomato_healthy": "Tomato leaf",
}

pd_folder_to_src_idx = {}
for pv_name, pd_name in PV2PD.items():
    if pd_name is None:
        continue
    if pv_name not in source_class_to_idx:
        continue
    pd_folder_to_src_idx[pd_name] = source_class_to_idx[pv_name]

print("[Info] PlantDoc folder -> source label index mapping")
for k, v in pd_folder_to_src_idx.items():
    print(f"  {k} -> {v}")


[Info] PlantDoc folder -> source label index mapping
  Bell_pepper leaf spot -> 0
  Bell_pepper leaf -> 1
  Potato leaf early blight -> 2
  Potato leaf late blight -> 3
  Tomato leaf bacterial spot -> 5
  Tomato Early blight leaf -> 6
  Tomato leaf late blight -> 7
  Tomato mold leaf -> 8
  Tomato Septoria leaf spot -> 9
  Tomato two spotted spider mites leaf -> 10
  Tomato leaf yellow virus -> 12
  Tomato leaf mosaic virus -> 13
  Tomato leaf -> 14


In [10]:

# =========================
# Dataset wrapper:
#   keep only mapped PlantDoc classes
#   relabel them into PlantVillage label indices
# =========================
class MappedImageFolderDataset(Dataset):
    def __init__(self, imagefolder_ds, folder_to_newlabel):
        self.base = imagefolder_ds
        self.folder_to_newlabel = folder_to_newlabel
        self.keep_indices = []

        for i, (_, y) in enumerate(self.base.samples):
            folder_name = self.base.classes[y]
            if folder_name in self.folder_to_newlabel:
                self.keep_indices.append(i)

        self.class_counts = {}
        for i in self.keep_indices:
            _, y = self.base.samples[i]
            folder_name = self.base.classes[y]
            self.class_counts[folder_name] = self.class_counts.get(folder_name, 0) + 1

        self.mapped_folder_names = sorted(self.class_counts.keys())

    def __len__(self):
        return len(self.keep_indices)

    def __getitem__(self, idx):
        real_idx = self.keep_indices[idx]
        x, old_y = self.base[real_idx]
        folder_name = self.base.classes[old_y]
        new_y = self.folder_to_newlabel[folder_name]
        return x, new_y


In [11]:

# =========================
# Load ONLY train_labeled and test
# =========================
pd_labeled_raw = datasets.ImageFolder(PD_LABELED_DIR, transform=train_tf)
pd_test_raw = datasets.ImageFolder(PD_TEST_DIR, transform=test_tf)

pd_labeled_ds = MappedImageFolderDataset(pd_labeled_raw, pd_folder_to_src_idx)
pd_test_ds = MappedImageFolderDataset(pd_test_raw, pd_folder_to_src_idx)

print("[Info] raw train_labeled classes:", pd_labeled_raw.classes)
print("[Info] raw test classes:", pd_test_raw.classes)

print("[Info] mapped train_labeled size:", len(pd_labeled_ds))
print("[Info] mapped test size:", len(pd_test_ds))
print("[Info] mapped train_labeled class counts:", pd_labeled_ds.class_counts)
print("[Info] mapped test class counts:", pd_test_ds.class_counts)

unmapped_labeled = sorted(set(pd_labeled_raw.classes) - set(pd_folder_to_src_idx.keys()))
unmapped_test = sorted(set(pd_test_raw.classes) - set(pd_folder_to_src_idx.keys()))
print("[Info] unmapped classes in train_labeled:", unmapped_labeled)
print("[Info] unmapped classes in test:", unmapped_test)

if len(pd_labeled_ds) == 0:
    raise RuntimeError("Mapped train_labeled dataset is empty. Check folder names and class mapping.")
if len(pd_test_ds) == 0:
    raise RuntimeError("Mapped test dataset is empty. Check folder names and class mapping.")


[Info] raw train_labeled classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato mold leaf', 'grape leaf', 'grape leaf black rot']
[Info] raw test classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato 

In [12]:

# =========================
# DataLoaders
# =========================
pd_labeled_loader = DataLoader(
    pd_labeled_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

pd_test_loader = DataLoader(
    pd_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

print("[Info] labeled batches:", len(pd_labeled_loader))
print("[Info] test batches:", len(pd_test_loader))


[Info] labeled batches: 2
[Info] test batches: 2


In [13]:

# =========================
# Build model and load baseline weights
# =========================
def build_model(name: str, num_classes: int, pretrained: bool = False):
    name = name.lower()
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    raise ValueError("Only resnet18/resnet50 supported.")

model = build_model(MODEL_NAME, num_classes=num_classes, pretrained=False)

if "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
elif "model" in ckpt:
    model.load_state_dict(ckpt["model"])
else:
    raise RuntimeError("Cannot find model weights inside baseline checkpoint.")

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))

print("[Info] Loaded baseline checkpoint successfully.")


[Info] Loaded baseline checkpoint successfully.


In [16]:

# =========================
# Train / Eval functions
# =========================
def run_train_epoch(loader):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(USE_AMP and device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / max(1, total), correct / max(1, total)

@torch.no_grad()
def confusion_matrix(pred, true, num_classes):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.long)
    for p, t in zip(pred, true):
        cm[t, p] += 1
    return cm

def macro_f1_from_cm(cm):
    f1s = []
    for c in range(cm.size(0)):
        tp = cm[c, c].item()
        fp = cm[:, c].sum().item() - tp
        fn = cm[c, :].sum().item() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        f1s.append(f1)

    return float(sum(f1s) / len(f1s)) if len(f1s) > 0 else 0.0

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds = []
    all_true = []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        pred = logits.argmax(1)

        total_loss += loss.item() * x.size(0)
        correct += (pred == y).sum().item()
        total += y.size(0)

        all_preds.extend(pred.cpu().tolist())
        all_true.extend(y.cpu().tolist())

    cm = confusion_matrix(all_preds, all_true, num_classes)
    macro_f1 = macro_f1_from_cm(cm)

    return {
        "loss": total_loss / max(1, total),
        "acc": correct / max(1, total),
        "macro_f1": macro_f1,
        "cm": cm
    }


In [ ]:

# =========================
# Fine-tune ONLY on train_labeled
# Evaluate on PlantDoc test
# =========================

EPOCHS_FINETUNE = 10 

finetune_best = os.path.join(OUT_DIR, "finetune_best.pt")
log_path = os.path.join(OUT_DIR, "finetune_log.json")
cm_path = os.path.join(OUT_DIR, "finetune_confusion_matrix.txt")

best_acc = -1.0
log = []

print("\n========== Fine-tune on PlantDoc/train_labeled ==========")
for epoch in range(1, EPOCHS_FINETUNE + 1):
    t0 = time.time()

    ft_train_loss, ft_train_acc = run_train_epoch(pd_labeled_loader)
    ev = evaluate(pd_test_loader)

    row = {
        "epoch": epoch,
        "ft_train_loss": float(ft_train_loss),
        "ft_train_acc": float(ft_train_acc),
        "pd_test_loss": float(ev["loss"]),
        "pd_test_acc": float(ev["acc"]),
        "pd_test_macro_f1": float(ev["macro_f1"]),
        "sec": float(time.time() - t0)
    }
    log.append(row)

    print(
        f"Epoch {epoch:03d}/{EPOCHS_FINETUNE} | "
        f"train_labeled loss {ft_train_loss:.4f} acc {ft_train_acc:.4f} | "
        f"PlantDoc test loss {ev['loss']:.4f} acc {ev['acc']:.4f} macroF1 {ev['macro_f1']:.4f} | "
        f"{time.time()-t0:.1f}s"
    )

    if ev["acc"] > best_acc:
        best_acc = ev["acc"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_acc": best_acc,
            "epoch": epoch,
            "classes": source_classes,
            "model_name": MODEL_NAME,
            "baseline_ckpt": BASELINE_CKPT,
        }, finetune_best)

with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2, ensure_ascii=False)

# final confusion matrix
final_ev = evaluate(pd_test_loader)
with open(cm_path, "w", encoding="utf-8") as f:
    f.write("Confusion Matrix (rows=true, cols=pred)\n")
    f.write(str(final_ev["cm"].tolist()) + "\n")
    f.write(f"Final Acc: {final_ev['acc']:.6f}\n")
    f.write(f"Final Macro-F1: {final_ev['macro_f1']:.6f}\n")

print("\n[Done] best PlantDoc test acc =", best_acc)
print("[Saved] finetune best checkpoint:", finetune_best)
print("[Saved] finetune log:", log_path)
print("[Saved] finetune confusion matrix:", cm_path)


========== Fine-tune on PlantDoc/train_labeled ==========
Epoch 001/10 | train_labeled loss 0.3807 acc 0.8947 | PlantDoc test loss 2.4827 acc 0.4216 macroF1 0.3189 | 3.6s
Epoch 002/10 | train_labeled loss 0.3084 acc 0.9474 | PlantDoc test loss 2.4664 acc 0.4020 macroF1 0.3048 | 3.4s
Epoch 003/10 | train_labeled loss 0.2547 acc 0.9474 | PlantDoc test loss 2.4433 acc 0.3922 macroF1 0.3052 | 3.7s
Epoch 004/10 | train_labeled loss 0.2180 acc 0.9684 | PlantDoc test loss 2.4360 acc 0.3824 macroF1 0.3012 | 4.8s
Epoch 005/10 | train_labeled loss 0.1697 acc 0.9579 | PlantDoc test loss 2.4189 acc 0.3725 macroF1 0.2940 | 3.6s
Epoch 006/10 | train_labeled loss 0.1047 acc 0.9895 | PlantDoc test loss 2.3863 acc 0.3922 macroF1 0.3059 | 3.5s
Epoch 007/10 | train_labeled loss 0.0964 acc 0.9895 | PlantDoc test loss 2.3515 acc 0.3824 macroF1 0.2988 | 3.6s
Epoch 008/10 | train_labeled loss 0.0688 acc 0.9895 | PlantDoc test loss 2.3312 acc 0.3922 macroF1 0.3013 | 3.6s
Epoch 009/10 | train_labeled loss 0.0